# Real-Time Emotion Recognition System

This notebook implements real-time facial emotion recognition using:
- **YOLOv8** for face detection
- **OpenCV** for video capture and display

## Features:
- Real-time webcam emotion detection
- Multiple face support
- FPS counter
- Confidence scores
- Color-coded emotion displays

## 1. Import Libraries and Setup

In [1]:
# Import required libraries
import cv2
import numpy as np
import tensorflow as tf
from ultralytics import YOLO
import time
import os
from collections import deque
import warnings
warnings.filterwarnings('ignore')

# Check versions
print(f"TensorFlow version: {tf.__version__}")
print(f"OpenCV version: {cv2.__version__}")
print(f"CUDA available: {tf.test.is_built_with_cuda()}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
from tensorflow.keras.applications.resnet_v2 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as eff_preprocess

TensorFlow version: 2.10.0
OpenCV version: 4.7.0
CUDA available: True
GPU available: True


## 2. Configuration

In [2]:
import os
os.environ["TORCH_LOAD_WEIGHTS_ONLY"] = "0"  # must run before torch.load gets called

import torch
from torch.serialization import add_safe_globals

from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules import Conv, C2f, Bottleneck, SPP, SPPF
from ultralytics.nn.modules.head import Detect
from torch.nn import Sequential

add_safe_globals([
    DetectionModel,
    Conv,
    C2f,
    Bottleneck,
    SPP,
    SPPF,
    Detect,
    Sequential,
])

In [3]:
import os
os.environ["TORCH_LOAD_WEIGHTS_ONLY"] = "0"  # must run before torch.load gets called
import torch
from torch.serialization import add_safe_globals
from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules import Conv, C2f, Bottleneck, SPP, SPPF   # this import shape works for 8.x releases
from ultralytics.nn.modules.head import Detect
from torch.nn import Sequential  # torch.nn.modules.container.Sequential alias

add_safe_globals([
    DetectionModel,
    Conv,
    C2f,
    Bottleneck,
    SPP,
    SPPF,
    Detect,
    Sequential,
])
YOLO_MODEL = "yolov8n.pt"  # or your own weight file
yolo_model = YOLO(YOLO_MODEL)
print(" YOLO model loaded successfully")

 YOLO model loaded successfully


In [4]:
# Configuration parameters
from pathlib import Path
MODEL_PATH = Path.cwd() / 'models' / 'best_model.h5'
# Emotion classes (must match training)
EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
# Color mapping for emotions (BGR format for OpenCV)
EMOTION_COLORS = {
    'angry': (0, 0, 255),      # Red
    'disgust': (0, 128, 0),    # Dark Green
    'fear': (255, 0, 255),     # Magenta
    'happy': (0, 255, 255),    # Yellow
    'neutral': (128, 128, 128),# Gray
    'sad': (255, 0, 0),        # Blue
    'surprise': (0, 165, 255)  # Orange
}
# Detection parameters
CONFIDENCE_THRESHOLD = 0.5  # YOLO confidence threshold
EMOTION_THRESHOLD = 0.3     # Minimum emotion confidence to display
SMOOTHING_WINDOW = 5        # Number of frames to average for smoothing
# Display parameters
WINDOW_NAME = 'Real-Time Emotion Recognition'
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.9
THICKNESS = 2
print("Configuration loaded successfully!")

Configuration loaded successfully!


## 3. Load Models

In [5]:
# Load emotion recognition model
from pathlib import Path
model_path = MODEL_PATH if isinstance(MODEL_PATH, Path) else Path(MODEL_PATH)
if not model_path.is_absolute():
    model_path = Path.cwd() / model_path

print("Loading emotion recognition model...")
try:
    emotion_model = tf.keras.models.load_model(
        str(model_path),
        compile=False
    )
    print(f" Emotion model loaded successfully from {model_path}")
    print(f"   name: {emotion_model.name}")
    print(f"   input_shape: {emotion_model.input_shape}")
except Exception as e:
    print(f" Error loading emotion model: {e}")
    print(f"Please verify the model file exists at: {model_path}")

# Load YOLO model for face detection
print("Loading YOLO face detection model...")
try:
    yolo_model = YOLO(YOLO_MODEL)
    print(f" YOLO model loaded successfully")
except Exception as e:
    print(f" Error loading YOLO model: {e}")

Loading emotion recognition model...
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3080 Ti, compute capability 8.6
 Emotion model loaded successfully from c:\Users\archi\New folder\ABSM Project 5\models\best_model.h5
   name: EfficientNetV2B0_FER
   input_shape: (None, 224, 224, 3)
Loading YOLO face detection model...
 YOLO model loaded successfully


## 4. Helper Functions

In [6]:
class EmotionDetector:
    def __init__(self, emotion_model, yolo_model, smoothing_window=5):
        self.emotion_model = emotion_model
        self.yolo_model = yolo_model
        self.face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
        self.emotion_history = {}  # Store emotion history for each face
        self.smoothing_window = smoothing_window
        
    def detect_faces(self, frame):
        """Detect faces: try Haar cascade first, fallback to YOLO person head crop"""
        faces = []
        # Haar cascade on grayscale
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        detections = self.face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(40, 40))
        if len(detections) > 0:
            # take largest face
            x, y, w, h = max(detections, key=lambda b: b[2]*b[3])
            pad_x = int(w * 0.1)
            pad_y = int(h * 0.1)
            x1 = max(0, x - pad_x)
            y1 = max(0, y - pad_y)
            x2 = min(frame.shape[1], x + w + pad_x)
            y2 = min(frame.shape[0], y + h + pad_y)
            faces.append((x1, y1, x2, y2, 1.0))
            return faces

        # Fallback to YOLO person detection
        results = self.yolo_model(frame, verbose=False)
        if results[0].boxes is not None:
            for box in results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
                confidence = box.conf[0].cpu().numpy()
                cls_id = int(box.cls[0].cpu().numpy()) if hasattr(box, 'cls') and box.cls is not None else 0
                if confidence > CONFIDENCE_THRESHOLD and cls_id == 0:
                    height = y2 - y1
                    width = x2 - x1
                    # Head-focused crop: top ~60% of box, central ~70% width
                    y1 = max(0, y1 - int(height * 0.05))
                    head_h = int(height * 0.6)
                    y2 = min(frame.shape[0], y1 + head_h)
                    cx = (x1 + x2) // 2
                    half_w = int(width * 0.35)
                    x1 = max(0, cx - half_w)
                    x2 = min(frame.shape[1], cx + half_w)
                    faces.append((x1, y1, x2, y2, confidence))
        return faces
    def preprocess_face(self, face_img):
        """Preprocess face image for the loaded emotion model."""
        target_h = self.emotion_model.input_shape[1] if self.emotion_model.input_shape[1] is not None else 48
        target_w = self.emotion_model.input_shape[2] if self.emotion_model.input_shape[2] is not None else 48
        target_c = self.emotion_model.input_shape[3] if len(self.emotion_model.input_shape) > 3 and self.emotion_model.input_shape[3] is not None else 3
        face_resized = cv2.resize(face_img, (target_w, target_h), interpolation=cv2.INTER_AREA)

        if target_c == 1:
            face_gray = cv2.cvtColor(face_resized, cv2.COLOR_BGR2GRAY).astype("float32")
            face = face_gray / 255.0
            face = np.expand_dims(face, axis=-1)
        else:
            face_rgb = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB).astype("float32")
            model_name = getattr(self.emotion_model, 'name', '').lower()
            if target_h <= 64:
                face = face_rgb / 255.0  # small CNN
            elif 'resnet' in model_name:
                face = resnet_preprocess(face_rgb)
            else:
                face = eff_preprocess(face_rgb)

        return np.expand_dims(face, axis=0)
    def predict_emotion(self, face_img, face_id=0):
        """Predict emotion from face image with smoothing"""
        # Preprocess face
        face_batch = self.preprocess_face(face_img)
        
        # Predict emotion
        predictions = self.emotion_model.predict(face_batch, verbose=0)[0]
        
        # Apply smoothing
        if face_id not in self.emotion_history:
            self.emotion_history[face_id] = deque(maxlen=self.smoothing_window)
        
        self.emotion_history[face_id].append(predictions)
        
        # Average predictions over history
        if len(self.emotion_history[face_id]) > 0:
            smoothed_predictions = np.mean(self.emotion_history[face_id], axis=0)
        else:
            smoothed_predictions = predictions
        
        # Get emotion label and confidence
        emotion_idx = np.argmax(smoothed_predictions)
        emotion_label = EMOTION_LABELS[emotion_idx]
        emotion_confidence = smoothed_predictions[emotion_idx]
        
        return emotion_label, emotion_confidence, smoothed_predictions
    
    def draw_results(self, frame, faces_emotions):
        """Draw bounding boxes and emotion labels on frame"""
        for (x1, y1, x2, y2), emotion, confidence, all_probs in faces_emotions:
            # Get color for emotion
            color = EMOTION_COLORS.get(emotion, (0, 255, 0))
            
            # Draw bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            
            # Prepare label with confidence
            label = f"{emotion}: {confidence:.2f}"
            
            # Calculate label position
            label_size, _ = cv2.getTextSize(label, FONT, FONT_SCALE, THICKNESS)
            label_y = y1 - 10 if y1 - 10 > 10 else y1 + label_size[1] + 10
            
            # Draw label background
            cv2.rectangle(frame, 
                         (x1, label_y - label_size[1] - 5),
                         (x1 + label_size[0], label_y + 5),
                         color, -1)
            
            # Draw label text
            cv2.putText(frame, label, (x1, label_y), 
                       FONT, FONT_SCALE, (0,0,0), THICKNESS)
            
            # Draw emotion probabilities bar (optional)
            self.draw_emotion_bar(frame, x1, y2 + 10, all_probs)
    
    def draw_emotion_bar(self, frame, x, y, probabilities, bar_width=150, bar_height=100):
        """Draw a small bar chart showing emotion probabilities"""
        if y + bar_height > frame.shape[0] or x + bar_width > frame.shape[1]:
            return  # Don't draw if it would go out of bounds
        
        # Create a small bar chart
        bar_height_per_emotion = bar_height // len(EMOTION_LABELS)
        
        for i, (emotion, prob) in enumerate(zip(EMOTION_LABELS, probabilities)):
            # Calculate bar dimensions
            bar_y = y + i * bar_height_per_emotion
            bar_length = int(prob * bar_width)
            
            # Get color
            color = EMOTION_COLORS.get(emotion, (0, 255, 0))
            
            # Draw bar
            cv2.rectangle(frame, (x, bar_y), 
                         (x + bar_length, bar_y + bar_height_per_emotion - 2),
                         color, -1)
            
            # Draw label
            label = f"{emotion[0].upper()}"
            cv2.putText(frame, label, (x + bar_width + 5, bar_y + bar_height_per_emotion - 5),
                       FONT, 0.4, color, 1)

print(" EmotionDetector class created successfully!")

 EmotionDetector class created successfully!


In [7]:
# FPS calculation helper
class FPSCounter:
    def __init__(self, avg_window=30):
        self.timestamps = deque(maxlen=avg_window)
        self.fps = 0
    
    def update(self):
        now = time.time()
        self.timestamps.append(now)
        if len(self.timestamps) > 1:
            self.fps = len(self.timestamps) / (self.timestamps[-1] - self.timestamps[0])
        return self.fps
    
    def get_fps(self):
        return self.fps

def draw_fps(frame, fps):
    """Draw FPS counter on frame"""
    fps_text = f"FPS: {fps:.1f}"
    cv2.putText(frame, fps_text, (10, 30), FONT, 1, (0, 255, 0), 2)

def draw_info_panel(frame):
    """Draw information panel with instructions"""
    instructions = [
        "Press 'q' or 'ESC' to quit",
        "Press 's' to save screenshot",
        "Press 'r' to reset emotion history"
    ]
    
    y_offset = frame.shape[0] - 80
    for i, text in enumerate(instructions):
        cv2.putText(frame, text, (10, y_offset + i*25), 
                   FONT, 0.6, (200, 200, 200), 1)

print(" Helper functions created!")

 Helper functions created!


## 5. Real-Time Emotion Recognition - Webcam

In [8]:
def run_webcam_emotion_recognition():
    """Run real-time emotion recognition on webcam feed"""
    
    # Initialize detector
    detector = EmotionDetector(emotion_model, yolo_model, SMOOTHING_WINDOW)
    fps_counter = FPSCounter()
    
    # Open webcam
    print("Opening webcam...")
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print(" Error: Could not open webcam")
        return
    
    # Set webcam properties for better performance
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    
    print(" Webcam opened successfully")
    print("Starting emotion recognition...")
    print("Press 'q' or 'ESC' to quit")
    
    frame_count = 0
    process_every_n_frames = 1  # Process every frame for smoother overlays
    last_faces_emotions = []
    
    while True:
        # Capture frame
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame")
            break
        
        # Flip frame horizontally for mirror effect
        frame = cv2.flip(frame, 1)
        
        # Process frame for emotion detection
        if frame_count % process_every_n_frames == 0:
            # Detect faces
            faces = detector.detect_faces(frame)
            
            # Process each face
            faces_emotions = []
            for i, (x1, y1, x2, y2, conf) in enumerate(faces):
                # Extract face region
                face_img = frame[y1:y2, x1:x2]
                
                if face_img.size > 0:  # Check if face region is valid
                    # Predict emotion
                    emotion, emotion_conf, all_probs = detector.predict_emotion(face_img, face_id=i)
                    
                    if emotion_conf > EMOTION_THRESHOLD:
                        faces_emotions.append(((x1, y1, x2, y2), emotion, emotion_conf, all_probs))
            
            # Draw results
            detector.draw_results(frame, faces_emotions)
            last_faces_emotions = faces_emotions
        else:
            # Use last detections to keep overlays visible on skipped frames
            detector.draw_results(frame, last_faces_emotions)
        
        # Update FPS
        fps = fps_counter.update()
        draw_fps(frame, fps)
        
        # Draw info panel
        draw_info_panel(frame)
        
        # Display frame
        cv2.imshow(WINDOW_NAME, frame)
        
        # Handle key press
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:  # 'q' or ESC
            break
        elif key == ord('s'):  # Save screenshot
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            filename = f"emotion_screenshot_{timestamp}.jpg"
            cv2.imwrite(filename, frame)
            print(f"Screenshot saved: {filename}")
        elif key == ord('r'):  # Reset emotion history
            detector.emotion_history.clear()
            print("Emotion history reset")
        
        frame_count += 1
    
    # Clean up
    cap.release()
    cv2.destroyAllWindows()
    print("\nWebcam closed successfully")

# Run the webcam emotion recognition
run_webcam_emotion_recognition()

Opening webcam...
 Webcam opened successfully
Starting emotion recognition...
Press 'q' or 'ESC' to quit

Webcam closed successfully


## 6. Process Video File

In [11]:
def process_video_file(video_path, output_path=None):
    """Process a video file for emotion recognition"""
    
    # Initialize detector
    detector = EmotionDetector(emotion_model, yolo_model, SMOOTHING_WINDOW)
    fps_counter = FPSCounter()
    
    # Open video
    print(f"Opening video: {video_path}")
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f" Error: Could not open video file: {video_path}")
        return
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f" Video properties:")
    print(f"  Resolution: {width}x{height}")
    print(f"  FPS: {fps}")
    print(f"  Total frames: {total_frames}")
    
    # Setup video writer if output path is provided
    out = None
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        print(f"  Output: {output_path}")
    
    print("\nProcessing video...")
    frame_count = 0
    process_every_n_frames = 1  # Process every frame for smoother overlays
    last_faces_emotions = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Process frame for emotion detection
        if frame_count % process_every_n_frames == 0:
            # Detect faces
            faces = detector.detect_faces(frame)
            
            # Process each face
            faces_emotions = []
            for i, (x1, y1, x2, y2, conf) in enumerate(faces):
                # Extract face region
                face_img = frame[y1:y2, x1:x2]
                
                if face_img.size > 0:
                    # Predict emotion
                    emotion, emotion_conf, all_probs = detector.predict_emotion(face_img, face_id=i)
                    
                    if emotion_conf > EMOTION_THRESHOLD:
                        faces_emotions.append(((x1, y1, x2, y2), emotion, emotion_conf, all_probs))
            
            # Draw results
            detector.draw_results(frame, faces_emotions)
            last_faces_emotions = faces_emotions
        else:
            detector.draw_results(frame, last_faces_emotions)
        
        # Update FPS
        fps_val = fps_counter.update()
        draw_fps(frame, fps_val)
        
        # Show progress
        progress = (frame_count / total_frames) * 100
        cv2.putText(frame, f"Progress: {progress:.1f}%", (10, 60), 
                   FONT, 0.7, (0, 255, 0), 2)
        
        # Write frame to output video
        if out:
            out.write(frame)
        
        # Display frame
        cv2.imshow('Video Emotion Recognition', frame)
        
        # Allow quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\nProcessing interrupted by user")
            break
        
        frame_count += 1
        
        # Print progress every 100 frames
        if frame_count % 100 == 0:
            print(f"Processed {frame_count}/{total_frames} frames ({progress:.1f}%)")
    
    # Clean up
    cap.release()
    if out:
        out.release()
    cv2.destroyAllWindows()
    
    print(f"\n Video processing completed!")
    print(f"   Total frames processed: {frame_count}")
    if output_path:
        print(f"   Output saved to: {output_path}")

# Example usage (uncomment and modify path to use):
# process_video_file('input_video.mp4', 'output_with_emotions.mp4')

## 7. Test with Single Image

In [12]:
def test_single_image(image_path):
    """Test emotion recognition on a single image"""
    
    # Initialize detector
    detector = EmotionDetector(emotion_model, yolo_model)
    
    # Read image
    print(f"Loading image: {image_path}")
    image = cv2.imread(image_path)
    
    if image is None:
        print(f" Error: Could not load image: {image_path}")
        return
    
    # Detect faces
    faces = detector.detect_faces(image)
    print(f"Found {len(faces)} face(s)")
    
    # Process each face
    faces_emotions = []
    for i, (x1, y1, x2, y2, conf) in enumerate(faces):
        # Extract face region
        face_img = image[y1:y2, x1:x2]
        
        if face_img.size > 0:
            # Predict emotion
            emotion, emotion_conf, all_probs = detector.predict_emotion(face_img, face_id=i)
            faces_emotions.append(((x1, y1, x2, y2), emotion, emotion_conf, all_probs))
            
            print(f"\nFace {i+1}:")
            print(f"  Detected emotion: {emotion}")
            print(f"  Confidence: {emotion_conf:.2f}")
            print(f"  All probabilities:")
            for emo, prob in zip(EMOTION_LABELS, all_probs):
                print(f"    {emo}: {prob:.3f}")
    
    # Draw results on image
    detector.draw_results(image, faces_emotions)
    
    # Display image
    cv2.imshow('Emotion Recognition Result', image)
    print("\nPress any key to close the image window")
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    # Save result
    output_path = image_path.replace('.', '_emotion_detected.')
    cv2.imwrite(output_path, image)
    print(f"\nResult saved to: {output_path}")

# Example usage (uncomment and modify path to use):
# test_single_image('test_image.jpg')

## 8. Cleanup and Summary

In [15]:
# Summary and cleanup
print("REAL-TIME EMOTION RECOGNITION SYSTEM READY!")

print("\n System Components:")
print(f"  - Emotion Model: {MODEL_PATH}")
print(f"  - Face Detection: YOLO {YOLO_MODEL}")
print(f"  - Emotion Classes: {', '.join(EMOTION_LABELS)}")
print(f"  - Input Size: 48x48 RGB")

print("\n Features:")
print("  - Real-time webcam processing")
print("  - Multiple face support")
print("  - Emotion confidence scores")
print("  - Color-coded emotion display")
print("  - FPS counter")
print("  - Prediction smoothing")
print("  - Video file processing")
print("  - Screenshot capability")

print("\n Usage:")
print("  1. Run cell 5 for webcam emotion recognition")
print("  2. Use process_video_file() for video files")
print("  3. Use test_single_image() for single images")

print("\n Expected Performance:")
print("  - 25-30 FPS on RTX 3080 Ti")
print("  - ~15-20ms per face for emotion prediction")
print("  - ~10-30ms for YOLO face detection")

print("\n Model Information:")
if 'emotion_model' in globals():
    print(f"  Model loaded: Yes")
    print(f"  Input shape: {emotion_model.input_shape}")
    print(f"  Parameters: {emotion_model.count_params():,}")
else:
    print(f"  Model loaded: No - Please run cell 3")

print("\n Ready for real-time emotion recognition!")

REAL-TIME EMOTION RECOGNITION SYSTEM READY!

 System Components:
  - Emotion Model: c:\Users\archi\New folder\ABSM Project 5\models\best_model.h5
  - Face Detection: YOLO yolov8n.pt
  - Emotion Classes: angry, disgust, fear, happy, neutral, sad, surprise
  - Input Size: 48x48 RGB

 Features:
  - Real-time webcam processing
  - Multiple face support
  - Emotion confidence scores
  - Color-coded emotion display
  - FPS counter
  - Prediction smoothing
  - Video file processing
  - Screenshot capability

 Usage:
  1. Run cell 5 for webcam emotion recognition
  2. Use process_video_file() for video files
  3. Use test_single_image() for single images

 Expected Performance:
  - 25-30 FPS on RTX 3080 Ti
  - ~15-20ms per face for emotion prediction
  - ~10-30ms for YOLO face detection

 Model Information:
  Model loaded: Yes
  Input shape: (None, 224, 224, 3)
  Parameters: 6,250,071

 Ready for real-time emotion recognition!
